In [50]:
import os
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types 
from typing import Optional, Dict, Any

import warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(level=logging.CRITICAL)


import litellm
import logging

logging.getLogger("LiteLLM").setLevel(logging.CRITICAL)
logging.getLogger("litellm").setLevel(logging.CRITICAL)
logging.disable(logging.CRITICAL)  

In [51]:
MODEL_GPT = "groq/openai/gpt-oss-120b"
llm = LiteLlm(model=MODEL_GPT, reasoning_format="hidden")

print(
    llm.llm_client.completion(
        model=llm.model,
        messages=[
            {
                "role": "user",
                "content": "Are you ready?"
            }
        ],
        tools=[]
    )
)
print("\nGroq is ready for use.")

ModelResponse(id='chatcmpl-34254f7b-7d5d-435b-bbed-09df063b5699', created=1788629591, model='openai/gpt-oss-120b', object='chat.completion', system_fingerprint='fp_49bfac06f1', choices=[Choices(finish_reason='stop', index=0, message=Message(content="Absolutely—I'm ready! How can I assist you today?", role='assistant', tool_calls=None, function_call=None, provider_specific_fields=None, reasoning='The user asks "Are you ready?" It\'s a simple question. We respond affirmatively, ask how can help. No policy issues.'))], usage=Usage(completion_tokens=49, prompt_tokens=75, total_tokens=124, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=28, rejected_prediction_tokens=None, text_tokens=None, image_tokens=None, video_tokens=None), prompt_tokens_details=None, queue_time=0.305217477, prompt_time=0.003115554, completion_time=0.101360611, total_time=0.104476165), usage_breakdown=None, x_groq={'id': 'req_01m1sa38kzef5rt9

In [52]:
from neo4j_for_adk import graphdb

In [53]:
neo4j_is_ready = graphdb.send_query("RETURN 'Neo4j is Ready!' as message")

print(neo4j_is_ready)

{'status': 'success', 'query_result': [{'message': 'Neo4j is Ready!'}]}


In [54]:
def say_hello(person_name: str) -> dict:
    """Formats a welcome message to a named person. 

    Args:
        person_name (str): the name of the person saying hello

    Returns:
        dict: A dictionary containing the results of the query.
              Includes a 'status' key ('success' or 'error').
              If 'success', includes a 'query_result' key with an array of result rows.
              If 'error', includes an 'error_message' key.
    """
    return graphdb.send_query("RETURN 'Hello to you, ' + $person_name AS reply",
    {
        "person_name": person_name
    })

In [55]:
print(say_hello("Arshad"))

{'status': 'success', 'query_result': [{'reply': 'Hello to you, Arshad'}]}


# friendly_cypher_agent

In [56]:
hello_agent = Agent(
    name="hello_agent_v1",
    model=llm,
    description="Has friendly chats with a user.",
    instruction="""You are a helpful assistant, chatting with a user. 
                Be polite and friendly, introducing yourself and asking who the user is. 

                If the user provides their name, use the 'say_hello' tool to get a custom greeting.
                If the tool returns an error, inform the user politely. 
                If the tool is successful, present the reply.
                """,
    tools=[say_hello], 
)

print(f"Agent '{hello_agent.name}' created.")

Agent 'hello_agent_v1' created.


# Run the Agent

To run an agent, we need some additional components namely an execution environment and memory.

## Create the Runner and SessionService


Let's assume we have a single user talking to the agent in a single session. Let's create this user, the session and the runner:
* `SessionService`: Responsible for managing conversation history and state for different users and sessions. The `InMemorySessionService` is a simple implementation that stores everything in memory, suitable for testing and simple applications. It keeps track of the messages exchanged.  
* `Runner`: The engine that orchestrates the interaction flow. It takes user input, routes it to the appropriate agent, manages calls to the LLM and tools based on the agent's logic, handles session updates via the `SessionService`, and yields events representing the progress of the interaction.

In [57]:
app_name = hello_agent.name + "_app"
user_id = hello_agent.name + "_user"
session_id = hello_agent.name + "_session_01"
    
session_service = InMemorySessionService()
await session_service.create_session(
    app_name=app_name,
    user_id=user_id,
    session_id=session_id
)
    
runner = Runner(
    agent=hello_agent,
    app_name=app_name,
    session_service=session_service
)

In [59]:
user_message = "Hello, I'm Arshad"
print(f"\n>>> User Message: {user_message}")

content = types.Content(role='user', parts=[types.Part(text=user_message)])

final_response_text = "Agent did not produce a final response." 


verbose = False
async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=content):
    if verbose:
        print(f"  [Event] Author: {event.author}, Type: {type(event).__name__}, Final: {event.is_final_response()}, Content: {event.content}")
    
    if event.is_final_response():
        if event.content and event.content.parts:
            final_response_text = event.content.parts[0].text 
        elif event.actions and event.actions.escalate: 
            final_response_text = f"Agent escalated: {event.error_message or 'No specific message.'}"
        break 

print(f"<<< Agent Response: {final_response_text}")


>>> User Message: Hello, I'm Arshad
<<< Agent Response: Hello again, Arshad! Nice to see you back. How can I assist you today?


In [61]:
class AgentCaller:
    """A simple wrapper class for interacting with an ADK agent."""
    
    def __init__(self, agent: Agent, runner: Runner, 
                 user_id: str, session_id: str):
        """Initialize the AgentCaller with required components."""
        self.agent = agent
        self.runner = runner
        self.user_id = user_id
        self.session_id = session_id


    def get_session(self):
        return self.runner.session_service.get_session(app_name=self.runner.app_name, user_id=self.user_id, session_id=self.session_id)

    
    async def call(self, user_message: str, verbose: bool = False):
        """Call the agent with a query and return the response."""
        print(f"\n>>> User Message: {user_message}")

        content = types.Content(role='user', parts=[types.Part(text=user_message)])

        final_response_text = "Agent did not produce a final response." 
        
        async for event in self.runner.run_async(user_id=self.user_id, session_id=self.session_id, new_message=content):
            if verbose:
                print(f"  [Event] Author: {event.author}, Type: {type(event).__name__}, Final: {event.is_final_response()}, Content: {event.content}")

            if event.is_final_response():
                if event.content and event.content.parts:
                    final_response_text = event.content.parts[0].text
                elif event.actions and event.actions.escalate:
                    final_response_text = f"Agent escalated: {event.error_message or 'No specific message.'}"
                break 

        print(f"<<< Agent Response: {final_response_text}")
        return final_response_text


In [62]:
async def make_agent_caller(agent: Agent, initial_state: Optional[Dict[str, Any]] = {}) -> AgentCaller:
    """Create and return an AgentCaller instance for the given agent."""
    app_name = agent.name + "_app"
    user_id = agent.name + "_user"
    session_id = agent.name + "_session_01"
    
    session_service = InMemorySessionService()
    await session_service.create_session(
        app_name=app_name,
        user_id=user_id,
        session_id=session_id,
        state=initial_state
    )
    
    runner = Runner(
        agent=agent,
        app_name=app_name,
        session_service=session_service
    )
    
    return AgentCaller(agent, runner, user_id, session_id)


In [63]:
hello_agent_caller = await make_agent_caller(hello_agent)

async def run_conversation():
    await hello_agent_caller.call("Hello I'm Arshad")

    await hello_agent_caller.call("I am excited")

await run_conversation()


>>> User Message: Hello I'm Arshad
<<< Agent Response: Hello, Arshad! It's great to meet you. How can I help you today?

>>> User Message: I am excited
<<< Agent Response: That's wonderful to hear! 😊 What’s got you feeling excited today?


## A Simple Multi-Agent Team \- Delegation for Greetings & Farewells 